## ⤵️ Imports

In [66]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import load_wine
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from random import randint
import pandas as pd

In [67]:
data = pd.read_csv('/content/drive/MyDrive/2026/5 -  Projeto de Bloco: Engenharia de Dados: Big Data/dataset_vinhos.csv', sep=';')

In [68]:
data

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,target,target_label,type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,0,baixa qualidade,red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,9.8,0,baixa qualidade,red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,9.8,0,baixa qualidade,red
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,9.8,1,alta qualidade,red
4,7.4,0.66,0.00,1.8,0.075,13.0,40.0,0.99780,3.51,0.56,9.4,0,baixa qualidade,red
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5315,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,1,alta qualidade,white
5316,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,0,baixa qualidade,white
5317,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,1,alta qualidade,white
5318,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,1,alta qualidade,white


## 💡 Modelagem de dados

In [69]:
data['type'] = data['type'].map(
    {
        0: 'white',
        1: 'red'
    }
)

In [70]:
y = data['target']
X = data.drop(columns=['target', 'target_label', 'type'])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [71]:
print(f'X_train: {X_train.shape}')
print(f'X_test: {X_test.shape}')
print(f'y_train: {y_train.shape}')
print(f'y_test: {y_test.shape}')

X_train: (4256, 11)
X_test: (1064, 11)
y_train: (4256,)
y_test: (1064,)


### KNN

In [72]:
knn = KNeighborsClassifier(n_neighbors=10)

model = Pipeline(
  steps=[
      ('scaler', StandardScaler()),
      ('pca', PCA(n_components=9)),
      ('classifier', knn)
  ]
)

model.fit(X_train, y_train)

proba_train = model.predict_proba(X_train)[:,1]
proba_test = model.predict_proba(X_test)[:,1]

param_grid = {
    'pca__n_components':[5, 10],
    'classifier__n_neighbors': [2, 4, 6, 8, 10],
    'classifier__weights': ['distance','uniform', 'balanced']
}

classifier = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring=['precision', 'recall','accuracy','roc_auc'],
    cv=10,
    refit='precision',
    return_train_score=True,
    n_jobs=-1
).fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
100 fits failed out of a total of 300.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
52 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py", line 662, in fit
    self._final_estimator.fit(Xt, y, **

In [73]:
print(classifier.best_params_)
print(classifier.best_score_)

{'classifier__n_neighbors': 2, 'classifier__weights': 'uniform', 'pca__n_components': 5}
0.8124198073267095


### Decision Tree

In [74]:
dtc = DecisionTreeClassifier()

model_dtc = Pipeline(
  steps=[
      ('scaler', StandardScaler()),
      ('pca', PCA(n_components=9)),
      ('classifier', dtc)
  ]
)

model_dtc.fit(X_train, y_train)

proba_train = model_dtc.predict_proba(X_train)[:,1]
proba_test = model_dtc.predict_proba(X_test)[:,1]

param_grid = {
    'pca__n_components':[5, 10],
    'classifier__max_depth': [2, 4, 6, 8, 10],
    'classifier__class_weight': ['distance','uniform', 'balanced']
}

classifier = GridSearchCV(
    estimator=model_dtc,
    param_grid=param_grid,
    scoring=['precision', 'recall','accuracy','roc_auc'],
    cv=10,
    refit='precision',
    return_train_score=True,
    n_jobs=-1
).fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
200 fits failed out of a total of 300.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
100 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py", line 662, in fit
    self._final_estimator.fit(Xt, y, *

In [75]:
print(classifier.best_params_)
print(classifier.best_score_)

{'classifier__class_weight': 'balanced', 'classifier__max_depth': 6, 'pca__n_components': 5}
0.8142867950179846


## 🔎 Grid-Search

In [76]:
param_grid = {
    'pca__n_components':[5, 10],
    'classifier__n_neighbors': [2, 4, 6, 8, 10],
    'classifier__weights': ['distance','uniform', 'balanced']
}

classifier = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring=['precision', 'recall','accuracy','roc_auc'],
    cv=10,
    refit='precision',
    return_train_score=True,
    n_jobs=-1
).fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
100 fits failed out of a total of 300.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
48 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py", line 662, in fit
    self._final_estimator.fit(Xt, y, **

In [77]:
classifier

GridSearchCV(cv=10,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('pca', PCA(n_components=9)),
                                       ('classifier',
                                        KNeighborsClassifier(n_neighbors=10))]),
             n_jobs=-1,
             param_grid={'classifier__n_neighbors': [2, 4, 6, 8, 10],
                         'classifier__weights': ['distance', 'uniform',
                                                 'balanced'],
                         'pca__n_components': [5, 10]},
             refit='precision', return_train_score=True,
             scoring=['precision', 'recall', 'accuracy', 'roc_auc'])

In [78]:
classifier.best_estimator_

Pipeline(steps=[('scaler', StandardScaler()), ('pca', PCA(n_components=5)),
                ('classifier', KNeighborsClassifier(n_neighbors=2))])

In [79]:
classifier.best_index_

np.int64(2)

In [80]:
classifier.best_params_

{'classifier__n_neighbors': 2,
 'classifier__weights': 'uniform',
 'pca__n_components': 5}

In [81]:
classifier.best_score_

np.float64(0.8124198073267095)

In [82]:
classifier.cv_results_.keys()

dict_keys(['mean_fit_time', 'std_fit_time', 'mean_score_time', 'std_score_time', 'param_classifier__n_neighbors', 'param_classifier__weights', 'param_pca__n_components', 'params', 'split0_test_precision', 'split1_test_precision', 'split2_test_precision', 'split3_test_precision', 'split4_test_precision', 'split5_test_precision', 'split6_test_precision', 'split7_test_precision', 'split8_test_precision', 'split9_test_precision', 'mean_test_precision', 'std_test_precision', 'rank_test_precision', 'split0_train_precision', 'split1_train_precision', 'split2_train_precision', 'split3_train_precision', 'split4_train_precision', 'split5_train_precision', 'split6_train_precision', 'split7_train_precision', 'split8_train_precision', 'split9_train_precision', 'mean_train_precision', 'std_train_precision', 'split0_test_recall', 'split1_test_recall', 'split2_test_recall', 'split3_test_recall', 'split4_test_recall', 'split5_test_recall', 'split6_test_recall', 'split7_test_recall', 'split8_test_recall

## 🔀 Random-Search

In [83]:
param_distributions = {
    'pca__n_components':[3, 5, 8, 10, 12],
    'classifier__n_neighbors': [randint(1, 20)]
}

classifier = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    scoring='precision',
    cv=10,
    n_iter=10,
    refit=True,
    return_train_score=True,
    n_jobs=-1
).fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 5 is smaller than n_iter=10. Running 5 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
10 fits failed out of a total of 50.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
10 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wr

In [84]:
classifier.best_params_

{'pca__n_components': 10, 'classifier__n_neighbors': 10}

In [85]:
classifier.best_score_

np.float64(0.7873065871309114)